# Project 12: Web Requests, Caching, Scraping and DataFrames

## Your Information

At the start of each assignment, you will need to provide us your name and the name of the partner you worked with for this assignment (if you had one). Double click on the cell below or click once and hit enter to edit it. Replace "First Last" with your first name and last name. Replace "None" with the first and last name of your partner if you had one for this assignment. We ask for this information so we don't accuse you of cheating when your code looks like your partner's.

Please keep these lines commented so they don't cause an error.

In [ ]:
# MY NAME: Wenyu
# My PARTNER'S NAME: None

## Imports

Every project will begin with some import statements. It's crucial that you run the cell below, otherwise we will not be able to grade your code and provide feedback to you.

In [2]:
# Please place all your import statements in this cell if you need to import any more modules for this lab.

# We have imported these modules for you:
import requests
import os
import json
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

import student_grader

# Set the LLM feedback language: 
# 1=English, 2=Mandarin Chinese, 3=Spanish, 4=Hindi, 5=Portuguese, 
# or use a string (e.g., "French")
student_grader.initialize(os.getcwd(), "p12", language=1)

LLM response will be generated in English.


<h2 style="color:red">Warning (Note on Academic Misconduct)</h2>

**IMPORTANT**: **P12 and P13 are two parts of the same data analysis.** You **cannot** switch project partners between these two projects. That is, if you partnered up with someone for P12, you have to sustain that partnership until the end of P13, or work on P13 alone. Now may be a good time to review [our course policies](https://cs220.cs.wisc.edu/f25/syllabus.html).

Under any circumstances, no more than two students are allowed to work together on a project. If your code is flagged by our code similarity detection tools, both partners will be responsible for sharing/copying the code, even if the code is shared/copied by one of the partners with/from other non-partner student(s). Note that each case of plagiarism will be reported to the Dean of Students and the students involved will receive a zero grade on the project.

## Lab Portion (12 Questions, 1 Function)

### Learning Objectives

In this lab, you will practice how to:

* use HTTP requests to download content from the internet,
* cache data onto your computer,
* construct and modify DataFrames to analyze datasets.

### Segment 1: File Handling with the `os` Module

#### Lab Question 1

Fetch the file `rankings.json` using the `requests` library from this URL:

`https://cs220.cs.wisc.edu/f25_projects/data/rankings.json`.

Make sure to call the appropriate function to **raise** an HTTPError if status code is not `200`. Then create a variable called `file_text` that saves the text of the response.

Points possible: 6.0

In [5]:
# compute and store the answer in the variable 'file_text'
url = "https://cs220.cs.wisc.edu/f25_projects/data/rankings.json"
response = requests.get(url)
response.raise_for_status()
file_text = response.text

# do NOT display the whole variable

In [6]:
student_grader.check("lab-q1", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for lab-q1...
Great job! You passed all test cases for this question.


True

#### Lab Question 2

Save the fetched `rankings.json` from the Lab Question 1 locally. 

Open a file named `rankings.json` in write mode using **UTF-8 encoding**, and write the contents of the variable `file_text` into it. Make sure to **close** the file afterwards (unless you used a `with` block to open the file).

Look at the contents of your `p12` folder. You should see a file called `rankings.json` appear there.

Points possible: 6.0

In [9]:
# write your code here
with open("rankings.json", "w", encoding="utf-8") as f:
    f.write(file_text)

In [10]:
student_grader.check("lab-q2", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for lab-q2...
Great job! You passed all test cases for this question.


True

### Lab Function 1: `download(url, filename)`

Now, you will implement a function `download` to download data from the internet and save it to a file locally. 

This function takes in two arguments: `url` and `filename`. The contents at the address pointed to by the `url` field should be saved into the file on your machine whose path is specified by `filename`. Remember that you can reuse the code you wrote above.

The naive version of function `download` described above has one big disadvantage: it downloads the data and creates the file *even if* the data has been downloaded and file has already been created. Fetching data from webpages takes both time and resources, so repeatedly downloading files that have been already downloaded is a **very bad** coding practice, and **must** be avoided.

Therefore, we need to ensure the `download` function implements *caching*. This means that **before** downloading the file from the internet, the function **must** check if the file already exists. If the file already exists, the function should return the message `"<filename> already exists!"` where `filename` is the argument. It should **not** make a request. Don't forget the space between `<filename>` and the following word. 

If the file **does not** exist, then you should download and save it in the file whose path is specified by `<filename>` and it should return the message `"<filename> created!"` where `filename` is the argument.

**Hint:** You can use the `os.path.exists` function to check if the `filename` already exists.

**Hint:** If you're struggling to pass the test, we've shown you the code that we are using to test your `download` function in Lab Question 3.

**Note:** The autograder will only check whether the file has been created, not the contents of the file. You will need to manually verify the contents of the file before proceeding.

Points possible: 16.0

In [13]:
# define the function 'download' here

def download(url, filename):
    if os.path.exists(filename):
        return filename + " already exists!"
    response = requests.get(url)
    response.raise_for_status()
    text = response.text
    with open(filename, "w", encoding="utf-8") as f:
        f.write(text)
    return filename + " created!"


In [14]:
student_grader.check("lab-download", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for lab-download...
Great job! You passed all test cases for this question.


True

#### Lab Question 3

Below is the code we used to check if your `download` function is correct.

The cell checks whether the `download` function checks existing files, and whether it correctly downloads files from the given address. An `assert` statement checks if a boolean expression is true and throws an error if not.

**Do NOT modify the code in the cell below**. Think about why the test code is written in this way, and ask a TA or PM if you're not sure.

Points possible: 6.0

In [17]:
rankings_url = "https://cs220.cs.wisc.edu/f25_projects/data/rankings.json"

# --- First download ---
# Check if the file exists
if os.path.exists("rankings.json"):
    # Delete the file
    os.remove("rankings.json")

return_result_1 = download(rankings_url, "rankings.json")
assert return_result_1 == "rankings.json created!", "Download 1st time failed"

# Get the file size
file_size_1 = os.path.getsize("rankings.json")
assert (file_size_1 > 0) == True, "File size after 1st download incorrect"

with open("rankings.json", "r", encoding="utf-8") as f:
    r = requests.get(rankings_url)
    r.raise_for_status()
    data = r.text
    assert f.read() == data, "File contents after 1st download mismatch"


# --- Second download ---
# Create an empty file
f = open("rankings.json", "w")
f.close()

return_result_2 = download(rankings_url, "rankings.json")
assert return_result_2 == "rankings.json already exists!", "Download 2nd time (file exists) failed"

file_size_2 = os.path.getsize("rankings.json")
assert file_size_2 == 0, "File size after 2nd download incorrect"


# --- Third download ---
os.remove("rankings.json")

return_result_3 = download(rankings_url, "rankings.json")
assert return_result_3 == "rankings.json created!", "Download 3rd time (after deletion) failed"

In [18]:
student_grader.check("lab-q3", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for lab-q3...
Great job! You passed all test cases for this question.


True

<h4 style="color:red">Warning</h4>

You **must** use this `download` function to download files during P12. This will ensure that you do not download the files each time you 'Restart & Run All'.

### Segment 2: DataFrame Operations

For this project, we will be analyzing statistics about world university rankings adapted from the
[Center for World University Rankings](https://cwur.org/). The `rankings.json` file was created by scraping content from pages on the linked website. 

We are going to use `pandas` throughout the lab and project to analyze this dataset.

### Task 2.1: Creating a DataFrame

#### Lab Question 4

Create a DataFrame `rankings` by reading the JSON file `rankings.json`.

The DataFrame `rankings` should look like this:

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Year</th>
      <th>World Rank</th>
      <th>Institution</th>
      <th>Country</th>
      <th>National Rank</th>
      <th>Education Rank</th>
      <th>Employability Rank</th>
      <th>Faculty Rank</th>
      <th>Research Rank</th>
      <th>Score</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>2023</td>
      <td>1</td>
      <td>Harvard University</td>
      <td>USA</td>
      <td>1</td>
      <td>1.0</td>
      <td>1.0</td>
      <td>1.0</td>
      <td>1.0</td>
      <td>100.0</td>
    </tr>
    <tr>
      <th>1</th>
      <td>2023</td>
      <td>2</td>
      <td>Massachusetts Institute of Technology</td>
      <td>USA</td>
      <td>2</td>
      <td>4.0</td>
      <td>12.0</td>
      <td>3.0</td>
      <td>9.0</td>
      <td>96.7</td>
    </tr>
    <tr>
      <th>2</th>
      <td>2023</td>
      <td>3</td>
      <td>Stanford University</td>
      <td>USA</td>
      <td>3</td>
      <td>11.0</td>
      <td>4.0</td>
      <td>2.0</td>
      <td>2.0</td>
      <td>95.2</td>
    </tr>
    <tr>
      <th>3</th>
      <td>2023</td>
      <td>4</td>
      <td>University of Cambridge</td>
      <td>United Kingdom</td>
      <td>1</td>
      <td>3.0</td>
      <td>25.0</td>
      <td>5.0</td>
      <td>11.0</td>
      <td>94.1</td>
    </tr>
    <tr>
      <th>4</th>
      <td>2023</td>
      <td>5</td>
      <td>University of Oxford</td>
      <td>United Kingdom</td>
      <td>2</td>
      <td>7.0</td>
      <td>27.0</td>
      <td>9.0</td>
      <td>4.0</td>
      <td>93.3</td>
    </tr>
  </tbody>
</table>

Points possible: 6.0

In [21]:
# do NOT remove this line
download("https://cs220.cs.wisc.edu/f25_projects/data/rankings.json", "rankings.json")
rankings = pd.read_json("rankings.json")

In [22]:
student_grader.check("lab-q4", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for lab-q4...
Great job! You passed all test cases for this question.


True

### Task 2.2: Counting Instances in a Dataframe

#### Lab Question 5

List all **unique** universities represented in the dataset.

As the dataset contains rankings for three different years, the same university may be featured multiple times. Find the names of the unique universities that are represented in the dataset.

First, extract just the names of the institutions as a pandas `Series`. Then, make a **list** of unique names called `institutions_list`. A `Series` can be easily typecast just like any other data type in Python. Order doesn't matter. 

Points possible: 6.0

In [25]:
# replace the ... with your code
institutions_series = rankings["Institution"]
institutions_list = list(set(institutions_series))


In [26]:
student_grader.check("lab-q5", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for lab-q5...
Great job! You passed all test cases for this question.


True

Now, let's find the country that is the 5th most represented in the DataFrame, and the number of times it's featured. Recall that `value_counts` enables us to count number of occurrences of unique values in a pandas `Series`.

#### Lab Question 6

Obtain the counts of all countries represented in the dataset.

Use the `value_counts` function on the `Country` column of `rankings` to create a pandas `Series` named `country_counts`, listing each country and the number of times it appears in the dataset.

Points possible: 6.0

In [29]:
# compute and store the answer in the variable 'country_counts'
# note that 'country_counts' should be of type 'pd.Series'
country_counts = rankings["Country"].value_counts()
country_counts

Country
China              984
USA                980
Japan              331
United Kingdom     274
France             220
                  ... 
Kuwait               3
Uruguay              3
Algeria              2
North Macedonia      2
Jamaica              1
Name: count, Length: 96, dtype: int64

In [30]:
student_grader.check("lab-q6", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for lab-q6...
Great job! You passed all test cases for this question.


True

#### Lab Question 7

Find the **5th** most represented country in the dataset.  

Use the `.index` attribute of the series `country_counts` to fetch the name of the 5th most represented country. Note that `country_counts` is **sorted** in *decreasing* order of the number of times each country appears in `rankings`. 

Then, use `.loc` to fetch the count of this country, and **typecast** this value to an `int`. Make sure to use the series `country_counts` defined in Lab Question 6.

**Hint**: The pandas `Series.index` works differently from the `.index` method you are familiar with for **lists**. `Series.index` takes in the numerical **index** of the element you want to access, and returns the **label** of the element. Then, you can pass the **label** to `.loc` to access the corresponding value in the `Series`.

Points possible: 6.0

In [33]:
fifth_country = country_counts.index[4]
fifth_count = int(country_counts.loc[fifth_country])

print(fifth_country, fifth_count)

France 220


In [34]:
student_grader.check("lab-q7", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for lab-q7...
France 220
Great job! You passed all test cases for this question.


True

### Task 2.3: Differentiating between `.loc` and `.iloc`

There are two ways to access values from a pandas `DataFrame`: `.iloc` and `.loc`. In the previous question, you used the `.loc` method to select a row based on its **index label**, the country name. If you wanted to access a row based on its **integer position**, you would instead use `.iloc`.

This distinction can be a confusing one, since oftentimes the index label and the integer position are the same. As an example, use both `.loc` and `.iloc` to fetch the first row in `rankings`.

In [35]:
first_row_iloc = rankings.iloc[0]
first_row_iloc

Year                                2023
World Rank                             1
Institution           Harvard University
Country                              USA
National Rank                          1
Education Rank                       1.0
Employability Rank                   1.0
Faculty Rank                         1.0
Research Rank                        1.0
Score                              100.0
Name: 0, dtype: object

In [36]:
first_row_loc = rankings.loc[0]
first_row_loc

Year                                2023
World Rank                             1
Institution           Harvard University
Country                              USA
National Rank                          1
Education Rank                       1.0
Employability Rank                   1.0
Faculty Rank                         1.0
Research Rank                        1.0
Score                              100.0
Name: 0, dtype: object

The results are exactly the same! This happens since the integer positions correspond to the pandas indices in the `rankings` DataFrame. **However, this will not always hold true** - as we see in the next question.

#### Lab Question 8

Use **boolean indexing** to create a `DataFrame` named `rankings_arg_bra` containing only the university rankings from *Argentina* or *Brazil*.

**Hint**: When implementing **boolean indexing** in `pandas`, the `or` operator is represented by `|` and the `and` operator is represented by `&`. Remember that you should separate conditions with parentheses when performing boolean indexing with multiple conditions.

Points possible: 6.0

In [39]:
# compute and store the answer in the variable 'rankings_arg_bra'
rankings_arg_bra = rankings[(rankings["Country"] == "Argentina") | (rankings["Country"] == "Brazil")]
# rankings_arg_bra.head() # uncomment to preview the DataFrame

In [40]:
student_grader.check("lab-q8", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for lab-q8...
Great job! You passed all test cases for this question.


True

Now, we will try to extract the **first** value in this new `DataFrame` using both `.iloc` and `.loc`. As you'll see, using `.loc` will not work the same way it did before. In fact, it will throw a **KeyError**. To verify, run the two cells below:

In [41]:
first_row_iloc = rankings_arg_bra.iloc[0]
first_row_iloc

Year                                     2023
World Rank                                109
Institution           University of São Paulo
Country                                Brazil
National Rank                               1
Education Rank                          497.0
Employability Rank                      381.0
Faculty Rank                            156.0
Research Rank                            78.0
Score                                    81.5
Name: 108, dtype: object

In [42]:
# uncomment the code below to see how .loc causes a KeyError

# first_row_loc = rankings_arg_bra.loc[0]
# first_row_loc

We see that using `.loc` now causes a **KeyError**.

`.loc[0]` tries to find the row with the **index label** 0. This causes an error because `rankings_arg_bra` has no **index label** 0, only an **integer position** 0. The **index label** of the first element is 104.

**When you’re done, comment out the code so it doesn’t cause an error.**

### Task 2.4: Sorting DataFrames

#### Lab Question 9

Sort the `rankings_arg_bra` DataFrame by country and then by national rank.  

The DataFrame `rankings_arg_bra` in Lab Question 8 is sorted by `World Rank`, with the result that universities from *Argentina* and *Brazil* are interleaved throughout the data. **Re-sort** the data to sort by `Country` so that all universities from *Argentina* appear **first** followed by universities from *Brazil*. Within each country, the universities should be **sorted** by their `National Rank`. 

Use the `sort_values` function of `pandas`. Remember - by default, `pandas` returns a **new** sorted `DataFrame` and does **not** modify the existing one.

Recall that `sort_values` takes an argument for the parameter `by` as the column name, based on which you want to do the sorting. If you want to use one column for primary sorting and another for secondary sorting, you can specify a **list** of column names.

Points possible: 6.0

In [45]:
# compute and store the answer in the variable 'sorted_rankings_arg_bra'

sorted_rankings_arg_bra = rankings_arg_bra.sort_values(by=["Country", "National Rank"])
# sorted_rankings_arg_bra.head() # uncomment to preview the DataFrame

In [46]:
student_grader.check("lab-q9", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for lab-q9...
Great job! You passed all test cases for this question.


True

### Task 2.5: Simplifying DataFrame to Track Ranking Changes

As we have seen, universities that have featured in the rankings for multiple years are featured repeatedly. To simplify comparisons, we want to feature each university once and remove all other metrics. 

Instead of simply ranking universities, we want to find the **absolute change** in universities' rankings between the years *2023* and *2024*. We are only interested in the absolute change and **not** whether the rank increased or decreased.  

#### Lab Question 10

Find the **absolute** difference in `World Rank` for *"University of Madras"* between the `Year` *2023* and *2024*.

First, let's attempt to measure the change for one particular university, *University of Madras*.

Points possible: 6.0

In [51]:
# compute and store the answer in the variable 'absolute_diff_madras'

# TODO: first find the ranking of "University of Madras" in the year 2023
# TODO: then find the ranking of "University of Madras" in the year 2024
# TODO: remember to use .iloc[0] to extract the value
# TODO: calculate the value of 'absolute_diff_madras'
rank_2023 = rankings[(rankings["Institution"] == "University of Madras") & (rankings["Year"] == 2023)]["World Rank"].iloc[0]
rank_2024 = rankings[(rankings["Institution"] == "University of Madras") & (rankings["Year"] == 2024)]["World Rank"].iloc[0]
absolute_diff_madras = abs(rank_2023 - rank_2024)
absolute_diff_madras

np.int64(18)

In [52]:
student_grader.check("lab-q10", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for lab-q10...
Great job! You passed all test cases for this question.


True

#### Lab Question 11

Create a `Series` with the absolute difference in ranks for *"University of Madras"* between *2023* and *2024*

To do this, first create a dictionary with the keys `Institution` and `Absolute Change`, and assign the corresponding values for *University of Madras*. You should use the `absolute_diff_madras` variable from the previous question. Then, convert this dictionary to a `Series`.

Points possible: 6.0

In [53]:
madras_dict = {"Institution": "University of Madras", "Absolute Change": absolute_diff_madras}
madras_series = pd.Series(madras_dict)
madras_series


Institution        University of Madras
Absolute Change                      18
dtype: object

In [54]:
student_grader.check("lab-q11", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for lab-q11...
Great job! You passed all test cases for this question.


True

#### Lab Question 12

Create a DataFrame `change_in_rankings` with just 2 columns, `Institution` and `Absolute Change`, where **each** university is only featured once. If the institution is **not** present in the rankings of **both** years, we will just ignore it.

The institutions should be **sorted** in *increasing* order of their **absolute change**. For institutions with the **same** absolute change, sort them *alphabetically* by their **names**.

**Warning:** Even if your code is optimal, this cell may take a few seconds to run. However, if it takes much longer than that (say, if it takes 30 seconds or longer), then you will **need** to optimize your code so it runs faster.

Points possible: 18.0

In [57]:
# One possible approach (you may use another approach if you prefer)

# TODO: initialize an empty list
# TODO: loop through the institutions in 'institutions_list' defined in Lab Question 4
    # TODO: create a new DataFrame that has rankings for only this institution
    #       (hint: Use boolean indexing on the "Institution" column in the `rankings` DataFrame)
    # TODO: create a list of years by casting the "Year" column of your new DataFrame to a list
    # TODO: skip institution if 2023 or 2024 are *not* in this list
        
    # TODO: extract the "World Rank" for both years from the new DataFrame 
    #       (remember to use .iloc[0] to extract the actual value)
    # TODO: find their absolute difference
       
    # TODO: make a dictionary where the keys are the strings “Institution” and “Absolute Change”
    #       and the values are the corresponding values you just found for this institution
    
    # TODO: append this dictionary to the list initialized in the first step

# TODO: convert the list of dicts to a pandas DataFrame called 'change_in_rankings'
rows = []
for inst in institutions_list:
    df_inst = rankings[rankings["Institution"] == inst]
    years = list(df_inst["Year"])
    if 2023 not in years or 2024 not in years:
        continue
    rank_2023 = df_inst[df_inst["Year"] == 2023]["World Rank"].iloc[0]
    rank_2024 = df_inst[df_inst["Year"] == 2024]["World Rank"].iloc[0]
    diff = abs(rank_2023 - rank_2024)
    rows.append({"Institution": inst, "Absolute Change": diff})

change_in_rankings = pd.DataFrame(rows)
change_in_rankings = change_in_rankings.sort_values(by=["Absolute Change", "Institution"])

# change_in_rankings.head() # uncomment to preview the DataFrame

In [58]:
student_grader.check("lab-q12", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for lab-q12...
Great job! You passed all test cases for this question.


True

### Lab Submission

Great work! You are now ready to submit lab-p12.
Submit your `p12.ipynb` on Gradescope to the **lab-p12** assignment, like usual. Remember that the grades for the lab portion of the project and the actual assignment grade are independent. You will submit the same notebook (at different levels of completion) to two different assignments, namely lab P12 to `lab-p12` and project P12 to `p12`.

## Project Portion (27 Questions, 1 Function, 2 Data Structures)

### Learning Objectives

In this project, you will demonstrate your ability to

* use `BeautifulSoup` to parse web pages and extract useful information, 
* read and write files,
* create and use `pandas DataFrames`,

Please go through the lab portion before working on this project. The lab introduces some useful techniques related to this project.

### Introduction

For this project, you're going to analyze World University Rankings!

Specifically, you're going to use Pandas to analyze various statistics of the top ranked universities across the world, over the last three years.

**Important Warning:** Do **not** download any of the other `json` or `html` files manually (you **must** write Python code to download these automatically, as in the lab portion). When we run the autograder, the other files such as `rankings.json`, `2023.html`, `2024.html`, `2025.html` will **not** be in the directory. So, unless your `p12.ipynb` downloads these files, the Gradescope autograder will **deduct** points from your public score. More details can be found in the **Setup** section of the project.

### Setup

For this project, we will be analyzing statistics about world university rankings adapted from [here](https://cwur.org/). These are the specific webpages that we extracted the data from:

* https://cwur.org/2023.php
* https://cwur.org/2024.php
* https://cwur.org/2025.php

Later in the project, you will be scraping these webpages and extracting the data yourself. Since we don't want all of you bombarding these webpages with requests, we have made snapshots of these webpages, and hosted them on GitLab. You can find the snapshots here:

* https://cs220.cs.wisc.edu/f25_projects/data/2023.html
* https://cs220.cs.wisc.edu/f25_projects/data/2024.html
* https://cs220.cs.wisc.edu/f25_projects/data/2025.html

We have also tweaked the snapshots a little, to streamline the process of data extraction for you. You will be extracting the data from these three html pages and analyzing them. However, to make the start of the project a little easier, we have already parsed the files for you! We have gathered the data from these html files, and collected them in a single json file, which can be found here:

* https://cs220.cs.wisc.edu/f25_projects/data/rankings.json

You will work with this json file for most of this project. At the end of this project, you will generate an identical json file by parsing the html files yourself.

### Project Requirements

You **may not** hardcode indices in your code. You **may not** manually download **any** files for this project, unless you are **explicitly** told to do so. For all other files, you **must** use the `download` function to download the files.

**Store** your final answer for each question in the **variable specified for each question**. This step is important because the grader scores your work by comparing the value of this variable against the correct answer.

For some of the questions, we'll ask you to write (then use) a function to compute the answer. If you compute the answer **without** creating the function we ask you to write, the Gradescope autograder will **deduct** points from your public score, even if the way you did it produced the correct answer.

#### Required Functions:
- `download`
- `parse_html`

In this project, you will also be required to define certain **data structures**. If you do not create these data structures exactly as specified, the Gradescope autograder will **deduct** points from your public score, even if the way you did it produced the correct answer.

#### Required Data Structures:
- `rankings`
- `year_2023_ranking_df`
- `year_2024_ranking_df`
- `year_2025_ranking_df`
- `institutions_df`

In addition, you are also **required** to follow the requirements below:
* Avoid using loops to iterate over pandas DataFrames and instead use boolean indexing.
* Do **not** use `.loc` to look up data in **DataFrames** or **Series** unless you are explicitly told to do so. You are **allowed** to use `iloc`.
* Do **not** use **absolute** paths such as `C://mdoescher//cs220//p12`. You may **only** use **relative paths**.
* Do **not** leave irrelevant output or test code that we didn't ask for.
* **Avoid** calling **slow** functions multiple times within a loop.
* Do **not** define multiple functions with the same name or define multiple versions of one function with different names. Just keep the best version.

### Segment 3: BeautifulSoup

As mentioned above, the `rankings.json` file was created by parsing HTML content on the Web, specifically the web pages listed below.

* https://cs220.cs.wisc.edu/f25_projects/data/2023.html
* https://cs220.cs.wisc.edu/f25_projects/data/2024.html
* https://cs220.cs.wisc.edu/f25_projects/data/2025.html

Now, let's write a function to do this ourselves. We will use the `BeautifulSoup` module to scrape web pages and extract information. 

### Project Data Structure 1: Download `2023.html`, `2024.html` and `2025.html`

Use the `download` function you previously created to download the contents of each of the URLs above and save them into files. Name the files `2023.html`, `2024.html` and `2025.html` based on the respective URL.

Points possible: 4.0

In [61]:
# use the 'download' function to download the data from the webpages and save the return values in 
# 'message_2023', 'message_2024', and 'message_2025' corresponding to:

# 'https://cs220.cs.wisc.edu/f25_projects/data/2023.html' → '2023.html'
# 'https://cs220.cs.wisc.edu/f25_projects/data/2024.html' → '2024.html'
# 'https://cs220.cs.wisc.edu/f25_projects/data/2025.html' → '2025.html'
message_2023 = download("https://cs220.cs.wisc.edu/f25_projects/data/2023.html", "2023.html")
message_2024 = download("https://cs220.cs.wisc.edu/f25_projects/data/2024.html", "2024.html")
message_2025 = download("https://cs220.cs.wisc.edu/f25_projects/data/2025.html", "2025.html")

print(message_2023)
print(message_2024)
print(message_2025)


2023.html already exists!
2024.html already exists!
2025.html already exists!


In [62]:
student_grader.check("download-html", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for download-html...
2023.html already exists!
2024.html already exists!
2025.html already exists!
Great job! You passed all test cases for this question.


True

Note that the cell above only checks whether the files were created and have a size greater than 0, and **not** whether they contain the correct data. You must check that yourself. Check your `p12` directory; it should now have files called `2023.html`, `2024.html` and `2025.html`. **Manually open** these files and confirm they contain the contents of the respective page:
* [2023.html](https://cs220.cs.wisc.edu/f25_projects/data/2023.html)
* [2024.html](https://cs220.cs.wisc.edu/f25_projects/data/2024.html)
* [2025.html](https://cs220.cs.wisc.edu/f25_projects/data/2025.html)

#### Project Question 1

Read the contents of the file `2023.html` using **UTF-8 encoding**.

Points possible: 2.0

In [65]:
# open "2023.html", read in the file content, and store it in the variable 'file_content_2023'.
with open("2023.html", "r", encoding="utf-8") as f:
    file_content_2023 = f.read()
# do NOT display the whole variable

In [66]:
student_grader.check("q1", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q1...
Great job! You passed all test cases for this question.


True

#### Project Question 2

Create a `BeautifulSoup` object instance using the variable `file_content_2023` defined in Project Question 1. 

Note: the grader for this question only test whether the variable you created is a `BeautifulSoup` object, not whether it has the correct value. We will check the correctness of this variable in the following questions.

Points possible: 2.0

In [69]:
soup_2023 = BeautifulSoup(file_content_2023, "html.parser")

# type(soup_2023) # uncomment this line to verify that `soup_2023` is a BeautifulSoup object

In [70]:
student_grader.check("q2", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q2...
Great job! You passed all test cases for this question.


True

#### Project Question 3

The webpage has a `table` containing all the data we're trying to extract. Write the code to **find** this element. Use the variable `soup_2023` defined in Project Question 2.

Note: the grader for this question only tests whether you found *a* tag from the `BeautifulSoup` object defined in Project Question 2. It does not check if you found the *right* tag. We will check the correctness of the table by finding its header cells in Project Question 4.

Points possible: 2.0

In [73]:
# find the table from the 'soup_2023' object and store it in the variable 'table_2023'

table_2023 = soup_2023.find("table")

# do NOT display the whole variable

In [74]:
student_grader.check("q3", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q3...
Great job! You passed all test cases for this question.


True

#### Project Question 4

Find all the table header using the `table_2023` defined in Project Question 3. 

Remember that the table header is represented by the `th` tag. You can use the `find_all` function to find all instances of the given tag in a `BeautifulSoup` object.

**Hint**: The **header** should be a **list** of elements that can be obtained by using the `get_text` method for each `th` element in the table. You may also find list comprehension useful here.

Points possible: 3.0

In [77]:
# compute and store the answer in the variable 'header_2023'
header_2023 = [th.get_text() for th in table_2023.find_all("th")]
header_2023

['World Rank',
 'Institution',
 'Location',
 'National Rank',
 'Education Rank',
 'Employability Rank',
 'Faculty Rank',
 'Research Rank',
 'Score']

In [78]:
student_grader.check("q4", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q4...
Great job! You passed all test cases for this question.


True

#### Project Question 5

Find all the data cells in the second row of `table_2023`. 

Using the `table_2023` defined in Project Question 3, locate the **second** row of the table (the **first** row is the **header**). Then, find all the data cells in that row and store their **text content** in a list.

**Hint**: Rows in an HTML table are represented by the `tr` tag, and data cells within a row are represented by the `td` tag.

Points possible: 3.0

In [81]:
second_row_cells = [td.get_text() for td in table_2023.find_all("tr")[1].find_all("td")]
second_row_cells

['1Top\xa00.1%',
 'Harvard University\n  ',
 'USA',
 '1',
 '1',
 '1',
 '1',
 '1',
 '100']

In [82]:
student_grader.check("q5", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q5...
Great job! You passed all test cases for this question.


True

You probably noticed that the `World Rank` (the first value in `second_row_cells`) is not in the expected format. This is an issue you will often encounter when parsing HTML files. In the next question, you will extract the numeric value from this text.

#### Project Question 6

Extract the **World Rank** of the second row. 

Using the list `second_row_cells` obtained in Project Question 5, extract the numeric **World Rank** from the first cell. Only keep the rank number (the part before the "Top" text) and store it as an integer.

Points possible: 3.0

In [89]:
s = second_row_cells[0]
i = 0
while i < len(s) and s[i].isdigit():
    i += 1
second_row_world_rank = int(s[:i])
second_row_world_rank


1

In [90]:
student_grader.check("q6", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q6...
Great job! You passed all test cases for this question.


True

#### Project Question 7

Use the values from `header_2023` and `second_row_cells` to create a **dictionary**.

Scrape the **second** row of the table, convert each value to the appropriate **data type**, and store the results in a **dictionary**. The **keys** of the dictionary **must** match the columns in the DataFrame. You may hardcode these keys, but it is more efficient to use the variable `header_2023` obtained in the previous question.

The required data types for each column are:

|**Column Name**|**Data Type**|
|---------------|-------------|
|`World Rank`|**int**|
|`Institution`|**str**|
|`Location`|**str**|
|`National Rank`|**int**|
|`Education Rank`|**int**|
|`Employability Rank`|**int**|
|`Faculty Rank`|**int**|
|`Research Rank`|**int**|
|`Score`|**float**|

**Note 1**: Extract the **World Rank** as a single int, just as you did in the Project Question 6.

**Note 2**: For all string values, remember to call the `strip` method to remove any extra whitespace. 

You can **compare** your output with the data in `rankings.json` to confirm whether you have parsed the file correctly (note that you do **not** yet have to implement the `Year` column in your **dictionary**).

Points possible: 5.0

In [93]:
second_row_dict = {}
for i in range(len(second_row_cells)):
    key = header_2023[i]
    value = second_row_cells[i].strip()

    if key == "World Rank":
        s = value
        j = 0
        while j < len(s) and s[j].isdigit():
            j += 1
        second_row_dict[key] = int(s[:j])

    elif key in ["National Rank", "Education Rank", "Employability Rank", "Faculty Rank", "Research Rank"]:
        second_row_dict[key] = int(value)

    elif key == "Score":
        second_row_dict[key] = float(value)

    else:
        second_row_dict[key] = value

second_row_dict


{'World Rank': 1,
 'Institution': 'Harvard University',
 'Location': 'USA',
 'National Rank': 1,
 'Education Rank': 1,
 'Employability Rank': 1,
 'Faculty Rank': 1,
 'Research Rank': 1,
 'Score': 100.0}

In [94]:
student_grader.check("q7", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q7...
Great job! You passed all test cases for this question.


True

#### Project Question 8

Build a list containing one dictionary per data row in the table for 2023.

Continuing from Project Question 7, scrape **all** rows in `table_2023`, **convert** each value to the appropriate **data type**, create a row **dictionary** for each row, and append all row dictionaries to a **list**. Follow the same data type casting rules and requirements used in Project Question 7.

**Important**:
* Some fields in the dataset have **missing** values represented as `"-"` or as `""`. Replace these missing values with `None` in your **dictionary**.
* Do **not** display the entire list, as it will be very long. You may print only the first few elements to verify that your parsed output matches the table shown in the `2023.html` file.

Points possible: 3.0

In [97]:
# build a list of row dictionaries using the algorithm in Project Question 7
# and store the result in 'rows_2023'
rows_2023 = []
for tr in table_2023.find_all("tr")[1:]:
    cells = [td.get_text().strip() for td in tr.find_all("td")]
    if not cells:
        continue
    row_dict = {}
    for i in range(len(cells)):
        key = header_2023[i]
        value = cells[i].strip()
        if value == "" or value == "-":
            row_dict[key] = None
        elif key == "World Rank":
            s = value
            j = 0
            while j < len(s) and s[j].isdigit():
                j += 1
            row_dict[key] = int(s[:j])
        elif key in ["National Rank", "Education Rank", "Employability Rank", "Faculty Rank", "Research Rank"]:
            row_dict[key] = int(value)
        elif key == "Score":
            row_dict[key] = float(value)
        else:
            row_dict[key] = value
    rows_2023.append(row_dict)

rows_2023[:3]


[{'World Rank': 1,
  'Institution': 'Harvard University',
  'Location': 'USA',
  'National Rank': 1,
  'Education Rank': 1,
  'Employability Rank': 1,
  'Faculty Rank': 1,
  'Research Rank': 1,
  'Score': 100.0},
 {'World Rank': 2,
  'Institution': 'Massachusetts Institute of Technology',
  'Location': 'USA',
  'National Rank': 2,
  'Education Rank': 4,
  'Employability Rank': 12,
  'Faculty Rank': 3,
  'Research Rank': 9,
  'Score': 96.7},
 {'World Rank': 3,
  'Institution': 'Stanford University',
  'Location': 'USA',
  'National Rank': 3,
  'Education Rank': 11,
  'Employability Rank': 4,
  'Faculty Rank': 2,
  'Research Rank': 2,
  'Score': 95.2}]

In [98]:
student_grader.check("q8", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q8...
Great job! You passed all test cases for this question.


True

### Project Function 1: `parse_html(filename)`

In Project Questions 1-8 (specifically, questions 1-4, 8), we successfully located the table inside `2023.html` and parsed the table into a list of dictionaries. Naturally, we would want to generalize this process into a function so that it can work on the other html files, not just `2023.html`. The function `parse_html` should take in a `filename` as **input** and **return** a **list** of **dictionaries**, with each **dictionary** representing a **row** in the dataset.

Additionally, we **also** want to include the **key** `Year` to all our **dictionaries**. The `Year` value is **not** present in the dataset, so you need extract this value from the `filename`.

Points possible: 5.0

In [101]:
def parse_html(filename):
    '''parse_html(filename) parses an HTML file and returns a list of dictionaries containing the tabular data'''
    year_str = os.path.basename(filename).split(".")[0]
    year = int(year_str)

    with open(filename, "r", encoding="utf-8") as f:
        text = f.read()

    soup = BeautifulSoup(text, "html.parser")
    table = soup.find("table")
    header = [th.get_text() for th in table.find_all("th")]

    rows = []
    for tr in table.find_all("tr")[1:]:
        cells = [td.get_text().strip() for td in tr.find_all("td")]
        if not cells:
            continue
        row_dict = {}
        for i in range(len(cells)):
            key = header[i]
            value = cells[i].strip()

            if value == "" or value == "-":
                row_dict[key] = None
            elif key == "World Rank":
                s = value
                j = 0
                while j < len(s) and s[j].isdigit():
                    j += 1
                row_dict[key] = int(s[:j])
            elif key in ["National Rank", "Education Rank", "Employability Rank", "Faculty Rank", "Research Rank"]:
                row_dict[key] = int(value)
            elif key == "Score":
                row_dict[key] = float(value)
            else:
                row_dict[key] = value

        row_dict["Year"] = year
        rows.append(row_dict)

    return rows


In [102]:
student_grader.check("parse_html", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for parse_html...
Great job! You passed all test cases for this question.


True

#### Project Question 9

Calculate the **average** `Score` of the **first** 5 institutions in the file `2023.html`.

Your output **must** be a **float** calculated by averaging the scores from the first 5 dictionaries in the file. You **must** use the `parse_html` function to parse the file, and **slice** the list such that you would only loop through the **first five** institutions. For each **dictionary** in the **list** you must use the `Score` key to get the score for that particular institution.

Points possible: 3.0

In [112]:
# compute and store the answer in the variable 'avg_top_5'
data_2023 = parse_html("2023.html")
avg_top_5 = sum(d["Score"] for d in data_2023[:5]) / 5
avg_top_5

95.86

In [113]:
student_grader.check("q9", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q9...
Great job! You passed all test cases for this question.


True

#### Project Question 10

Parse the contents of the **three** files `2023.html`, `2024.html`, and `2025.html` and combine them to create a **single** file named `my_rankings.json`.

You **must** create a **file** named `my_rankings.json` in your current directory.

**Hints:**
1. Using the logic from the question above, combine the data from these three files into a single list of dicts, and write it into the file `"my_rankings.json"`.
2. You can use the `write_json` function that was introduced in lecture.

Points possible: 3.0

In [108]:
# the 'write_json' function from lecture has been provided for you here
def write_json(path, data):
    with open(path, 'w', encoding = "utf-8") as f:
        json.dump(data, f, indent = 2)
        
# define a variable called `my_rankings` that contains the combined contents of the individual parsed html files
# `my_rankings` should be a list of dictionaries
my_rankings = parse_html("2023.html") + parse_html("2024.html") + parse_html("2025.html")
write_json("my_rankings.json", my_rankings)

# write the contents of `my_rankings` into 'my_rankings.json'

In [109]:
student_grader.check("q10", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q10...
Great job! You passed all test cases for this question.


True

### Segment 4: Exploring `rankings.json`

#### Project Question 11

Identify the number of **countries** in the `rankings` DataFrame created in Lab Question 4.

Your output **must** be an **int** representing the number of *unique* countries in the dataset.

Points possible: 2.0

In [116]:
# compute and store the answer in the variable 'num_countries'
num_countries = rankings["Country"].nunique()
num_countries


96

In [117]:
student_grader.check("q11", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q11...
Great job! You passed all test cases for this question.


True

#### Project Question 12

Generate a `DataFrame` containing **all** the columns of the **best-ranked** institutions based on `World Rank` across all the years.

Your output **must** be a pandas `DataFrame` with 3 rows and 10 columns. It **must** contain all the data for the institutions with `World Rank` of *1*. It **must** look like this:

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Year</th>
      <th>World Rank</th>
      <th>Institution</th>
      <th>Country</th>
      <th>National Rank</th>
      <th>Education Rank</th>
      <th>Employability Rank</th>
      <th>Faculty Rank</th>
      <th>Research Rank</th>
      <th>Score</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>2023</td>
      <td>1</td>
      <td>Harvard University</td>
      <td>USA</td>
      <td>1</td>
      <td>1.0</td>
      <td>1.0</td>
      <td>1.0</td>
      <td>1.0</td>
      <td>100.0</td>
    </tr>
    <tr>
      <th>2000</th>
      <td>2024</td>
      <td>1</td>
      <td>Harvard University</td>
      <td>USA</td>
      <td>1</td>
      <td>1.0</td>
      <td>1.0</td>
      <td>1.0</td>
      <td>1.0</td>
      <td>100.0</td>
    </tr>
    <tr>
      <th>4000</th>
      <td>2025</td>
      <td>1</td>
      <td>Harvard University</td>
      <td>USA</td>
      <td>1</td>
      <td>1.0</td>
      <td>1.0</td>
      <td>1.0</td>
      <td>1.0</td>
      <td>100.0</td>
    </tr>
  </tbody>
</table>

Points possible: 2.0

In [120]:
# compute and store the answer in the variable 'best_ranked'
best_ranked = rankings[rankings["World Rank"] == 1]
best_ranked


,Year,World Rank,Institution,Country,National Rank,Education Rank,Employability Rank,Faculty Rank,Research Rank,Score
0,2023,1,Harvard University,USA,1,1.0,1.0,1.0,1.0,100.0
2000,2024,1,Harvard University,USA,1,1.0,1.0,1.0,1.0,100.0
4000,2025,1,Harvard University,USA,1,1.0,1.0,1.0,1.0,100.0


In [121]:
student_grader.check("q12", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q12...
Great job! You passed all test cases for this question.


True

#### Project Question 13

Generate a pandas `DataFrame` containing **all** the statistics of *University of Wisconsin–Madison*.

**Hint**: The `–` symbol in the text above is not the regular hyphen (`-`) symbol. It is recommended that you just *copy/paste* the string `'University of Wisconsin–Madison'` into your code instead of typing it yourself.

Your output **must** be a pandas **DataFrame** with 3 rows and 10 columns. It **must** look like this:

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Year</th>
      <th>World Rank</th>
      <th>Institution</th>
      <th>Country</th>
      <th>National Rank</th>
      <th>Education Rank</th>
      <th>Employability Rank</th>
      <th>Faculty Rank</th>
      <th>Research Rank</th>
      <th>Score</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>27</th>
      <td>2023</td>
      <td>28</td>
      <td>University of Wisconsin–Madison</td>
      <td>USA</td>
      <td>20</td>
      <td>36.0</td>
      <td>102.0</td>
      <td>30.0</td>
      <td>41.0</td>
      <td>87.0</td>
    </tr>
    <tr>
      <th>2027</th>
      <td>2024</td>
      <td>28</td>
      <td>University of Wisconsin–Madison</td>
      <td>USA</td>
      <td>20</td>
      <td>40.0</td>
      <td>110.0</td>
      <td>31.0</td>
      <td>40.0</td>
      <td>87.0</td>
    </tr>
    <tr>
      <th>4029</th>
      <td>2025</td>
      <td>30</td>
      <td>University of Wisconsin–Madison</td>
      <td>USA</td>
      <td>20</td>
      <td>38.0</td>
      <td>115.0</td>
      <td>34.0</td>
      <td>45.0</td>
      <td>86.8</td>
    </tr>
  </tbody>
</table>

Points possible: 2.0

In [124]:
uw_madison = rankings[rankings["Institution"] == "University of Wisconsin–Madison"]
uw_madison


,Year,World Rank,Institution,Country,National Rank,Education Rank,Employability Rank,Faculty Rank,Research Rank,Score
27,2023,28,University of Wisconsin–Madison,USA,20,36.0,102.0,30.0,41.0,87.0
2027,2024,28,University of Wisconsin–Madison,USA,20,40.0,110.0,31.0,40.0,87.0
4029,2025,30,University of Wisconsin–Madison,USA,20,38.0,115.0,34.0,45.0,86.8


In [125]:
student_grader.check("q13", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q13...
Great job! You passed all test cases for this question.


True

#### Project Question 14

What was the `National Rank` of the *University of Wisconsin–Madison* in the `Year` *2023*?

Your output **should** be an **int**. You **must** use **Boolean indexing** on the variable `uw_madison` (from the previous question) to answer this question.

**Hint:** Use Boolean indexing on the DataFrame `uw_madison` to find the data for the year *2023*. You may then extract the `National Rank` column from the subset DataFrame. Finally, use `.iloc` to look up the value in the DataFrame which contains only one row and one column.

Points possible: 2.0

In [128]:
# compute and store the answer in the variable 'uw_madison_nat_rank'
uw_madison_nat_rank = uw_madison[uw_madison["Year"] == 2023]["National Rank"].iloc[0]
uw_madison_nat_rank


np.int64(20)

In [129]:
student_grader.check("q14", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q14...
Great job! You passed all test cases for this question.


True

#### Project Question 15

What is the **average** `Score` of the *University of Wisconsin–Madison*?

Your output **must** be a **float**. You **should** use the variable `uw_madison` to answer this question.

**Hint:** You **must** extract the `Score` column of the `DataFrame` `uw_madison` as a `Series`. You can find the **average** of  all the scores in a `Series` with the `Series.mean` function.

Points possible: 2.0

In [132]:
# compute and store the answer in the variable 'uw_madison_avg_score'
uw_madison_avg_score = uw_madison["Score"].mean()
uw_madison_avg_score


np.float64(86.93333333333334)

In [133]:
student_grader.check("q15", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q15...
Great job! You passed all test cases for this question.


True

#### Project Question 16

Generate a pandas `DataFrame` containing **all** the columns of universities from the `Country` *Singapore* in the `Year` *2025*.

Your output **must** be a pandas **DataFrame** with 4 rows and 10 columns. It **must** look like this:

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Year</th>
      <th>World Rank</th>
      <th>Institution</th>
      <th>Country</th>
      <th>National Rank</th>
      <th>Education Rank</th>
      <th>Employability Rank</th>
      <th>Faculty Rank</th>
      <th>Research Rank</th>
      <th>Score</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>4079</th>
      <td>2025</td>
      <td>80</td>
      <td>National University of Singapore</td>
      <td>Singapore</td>
      <td>1</td>
      <td>375.0</td>
      <td>183.0</td>
      <td>NaN</td>
      <td>32.0</td>
      <td>82.9</td>
    </tr>
    <tr>
      <th>4131</th>
      <td>2025</td>
      <td>132</td>
      <td>Nanyang Technological University</td>
      <td>Singapore</td>
      <td>2</td>
      <td>NaN</td>
      <td>848.0</td>
      <td>NaN</td>
      <td>61.0</td>
      <td>80.8</td>
    </tr>
    <tr>
      <th>4891</th>
      <td>2025</td>
      <td>892</td>
      <td>Singapore University of Technology and Design</td>
      <td>Singapore</td>
      <td>3</td>
      <td>NaN</td>
      <td>NaN</td>
      <td>NaN</td>
      <td>853.0</td>
      <td>71.3</td>
    </tr>
    <tr>
      <th>5216</th>
      <td>2025</td>
      <td>1217</td>
      <td>Singapore Management University</td>
      <td>Singapore</td>
      <td>4</td>
      <td>NaN</td>
      <td>NaN</td>
      <td>NaN</td>
      <td>1170.0</td>
      <td>69.4</td>
    </tr>
  </tbody>
</table>

Points possible: 3.0

In [136]:
# compute and store the answer in the variable 'singapore_inst'
singapore_inst = rankings[(rankings["Country"] == "Singapore") & (rankings["Year"] == 2025)]
singapore_inst


,Year,World Rank,Institution,Country,National Rank,Education Rank,Employability Rank,Faculty Rank,Research Rank,Score
4079,2025,80,National University of Singapore,Singapore,1,375.0,183.0,NaN,32.0,82.9
4131,2025,132,Nanyang Technological University,Singapore,2,NaN,848.0,NaN,61.0,80.8
4891,2025,892,Singapore University of Technology and Design,Singapore,3,NaN,NaN,NaN,853.0,71.3
5216,2025,1217,Singapore Management University,Singapore,4,NaN,NaN,NaN,1170.0,69.4


In [137]:
student_grader.check("q16", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q16...
Great job! You passed all test cases for this question.


True

#### Project Question 17

In the `Year` *2025*, what was the **best-ranked** institution in the `Country` *Germany*?

Your output **must** be a **string** representing the **name** of this institution.

**Hint:** The best-ranked institution in *Germany* is the institution from Germany with a `National Rank` of *1*.

Points possible: 3.0

In [142]:
# compute and store the answer in the variable 'german_best_name'
german_best_name = rankings[(rankings["Country"] == "Germany") & (rankings["Year"] == 2025) & (rankings["National Rank"] == 1)]["Institution"].iloc[0]
german_best_name


'Ludwig Maximilian University of Munich'

In [143]:
student_grader.check("q17", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q17...
Great job! You passed all test cases for this question.


True

#### Project Question 18

In the `Year` *2025*, list **all** the institutions in the *USA* that were ranked **better** than the best-ranked institution in *Germany*.

Your output **must** be a **list** containing the **names** of all universities from *USA* with a **better** `World Rank` than the institution `german_best_name` in the `Year` *2025*. By **better** ranked, we refer to institutions with a **lower** value under the `World Rank` column.

**Hint:** You could store the entire row of the best ranked institution from Germany in a different variable in Project Question 17, and use it to extract its `World Rank`. You could go back to your answer for Project Question 17, and edit it slightly to do this.

Points possible: 4.0

In [144]:
# compute and store the answer in the variable 'us_better_than_german_best'
german_best_rank = rankings[(rankings["Institution"] == german_best_name) & (rankings["Year"] == 2025)]["World Rank"].iloc[0]
us_better_than_german_best = list(rankings[(rankings["Country"] == "USA") & (rankings["Year"] == 2025) & (rankings["World Rank"] < german_best_rank)]["Institution"])
us_better_than_german_best


['Harvard University',
 'Massachusetts Institute of Technology',
 'Stanford University',
 'Princeton University',
 'University of Pennsylvania',
 'Columbia University',
 'Yale University',
 'University of Chicago',
 'California Institute of Technology',
 'University of California, Berkeley',
 'Cornell University',
 'Northwestern University',
 'University of Michigan, Ann Arbor',
 'University of California, Los Angeles',
 'Johns Hopkins University',
 'Duke University',
 'University of Illinois at Urbana–Champaign',
 'New York University',
 'University of Washington',
 'University of Wisconsin–Madison',
 'University of California, San Diego',
 'University of Texas at Austin',
 'University of California, San Francisco',
 'University of North Carolina at Chapel Hill',
 'Dartmouth College',
 'Washington University in St. Louis',
 'University of Minnesota - Twin Cities']

In [145]:
student_grader.check("q18", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q18...
Great job! You passed all test cases for this question.


True

#### Project Question 19

What is the **best-ranked** institution based on `Education Rank` in *China* for the `Year` *2025*?

Your output **must** be a **string** representing the **name** of this institution. You may **assume** there is only one institution satisfying these requirements. By the **best-ranked** institution, we refer to the institution with the **least** value under the `Education Rank` column.

**Hint:** You can find the **minimum** value in a `Series` with the `Series.min` method. You can find the documentation [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.min.html) or by executing the line `help(pd.Series.min)` in a separate cell below.

Points possible: 4.0

In [148]:
# compute and store the answer in the variable 'china_best_qoe',
china_2025 = rankings[(rankings["Country"] == "China") & (rankings["Year"] == 2025)]
min_edu = china_2025["Education Rank"].min()
china_best_qoe = china_2025[china_2025["Education Rank"] == min_edu]["Institution"].iloc[0]
china_best_qoe


'Nanjing Institute of Technology'

In [149]:
student_grader.check("q19", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q19...
Great job! You passed all test cases for this question.


True

#### Project Question 20

What are the **top** *five* **best-ranked** institutions based on `Research Rank` in *India* for the `Year` *2025*?

Your output **must** be a **list** of institutions **sorted** in *increasing* order of their `Research Rank`.

**Hint:** For sorting a DataFrame based on the values of a particular column, you can use the `DataFrame.sort_values(by="column_name")` method (where `column_name` is the column on which you want to sort). You can find the documentation [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.dataframe.sort_values.html) or by executing the line `help(pd.DataFrame.sort_values)` in a separate cell below.

Points possible: 4.0

In [152]:
# compute and store the answer in the variable 'india_best_research'
india_2025 = rankings[(rankings["Country"] == "India") & (rankings["Year"] == 2025)]
india_best_research = list(india_2025.sort_values(by="Research Rank").head(5)["Institution"])
india_best_research


['Indian Institute of Science',
 'Indian Institute of Technology Bombay',
 'Indian Institute of Technology Madras',
 'Tata Institute of Fundamental Research',
 'Indian Institute of Technology Delhi']

In [153]:
student_grader.check("q20", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q20...
Great job! You passed all test cases for this question.


True

### Segment 5: Analyzing Data Across the Years

For the next few questions, we will be analyzing how the rankings of the institutions change across the three years in the dataset. As you might have already noticed, the list of institutions in each year's rankings are different. As a result, for several institutions in the dataset, we do not have the rankings for all three years. Since it will be more challenging to analyze such institutions, we will simply skip them.

#### Project Question 21

How **many** institutions have rankings for **all** three years?

Your output **must** be an **integer**. To get started, you have been provided with a code snippet below.

**Hint:** You could make **sets** of the institutions that appear in each `DataFrame`, and find their **intersection**.

Points possible: 5.0

In [160]:
# compute and store the answer in the variable 'num_institutions_2023_2024_2025'

year_2023_ranking_df = rankings[rankings["Year"] == 2023]
year_2024_ranking_df = rankings[rankings["Year"] == 2024]
year_2025_ranking_df = rankings[rankings["Year"] == 2025]

# TODO: make sets of the institutions in each of the three years
institutions_2023 = set(year_2023_ranking_df["Institution"])


institutions_2024 = set(year_2024_ranking_df["Institution"])
institutions_2025 = set(year_2025_ranking_df["Institution"])

institutions_2023_2024_2025 = institutions_2023 & institutions_2024 & institutions_2025
num_institutions_2023_2024_2025 = len(institutions_2023_2024_2025)

num_institutions_2023_2024_2025


1893

In [161]:
student_grader.check("q21", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q21...
Great job! You passed all test cases for this question.


True

### Project Data Structure 2: `institutions_df`

You are now going to create a new `DataFrame` with the **unique** institutions which have featured in the rankings for **all** three years, along with their `World Rank` across the three years. The `DataFrame` **must** have the following four columns - `'Institution'`, `'2023 ranking'`, `'2024 ranking'`, and `'2025 ranking'`. The `DataFrame` **must** be sorted alphabetically by the `Institution` name.

Points possible: 5.0

In [164]:
rows = []
for inst in institutions_2023_2024_2025:
    r2023 = year_2023_ranking_df[year_2023_ranking_df["Institution"] == inst]["World Rank"].iloc[0]
    r2024 = year_2024_ranking_df[year_2024_ranking_df["Institution"] == inst]["World Rank"].iloc[0]
    r2025 = year_2025_ranking_df[year_2025_ranking_df["Institution"] == inst]["World Rank"].iloc[0]
    rows.append({
        "Institution": inst,
        "2023 ranking": r2023,
        "2024 ranking": r2024,
        "2025 ranking": r2025
    })

institutions_df = pd.DataFrame(rows).sort_values(by="Institution")


In [165]:
student_grader.check("institutions_df", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for institutions_df...
Great job! You passed all test cases for this question.


True

#### Project Question 22

Between the years *2024* and *2025*, **list** the institutions which have seen an **improvement** in their `World Rank` by strictly **more than** *200* ranks.

Your output **must** be a **list** of institution names. **Order** does **not** matter. You **must** use the DataFrame `institutions_df` to answer this question.

**Hints:**

1. In pandas, subtraction of two columns can be simply done using the subtraction operator (`-`). For example,
``` python
df["difference"] = df["column1"] - df["column2"]
```
will create a *new column* `difference` with the difference of the values from the columns `column1` and `column2`.
2. Note that an *improved* ranking means that the `World Rank` has *decreased*.

Points possible: 4.0

In [168]:
# compute and store the answer in the variable 'improved_institutions'
diff = institutions_df["2024 ranking"] - institutions_df["2025 ranking"]
improved_institutions = list(institutions_df[diff > 200]["Institution"])
improved_institutions

['Beijing Technology and Business University',
 'Chengdu University',
 'Chengdu University of Traditional Chinese Medicine',
 'Dalian Polytechnic University',
 'East China Jiaotong University',
 'Foshan University',
 'Hubei University of Technology',
 'Ioffe Physical-Technical Institute of the Russian Academy of Sciences',
 'Nanjing University of Finance & Economics',
 'Near East University',
 'Prince Sattam Bin Abdulaziz University',
 'Qingdao University of Technology',
 'Suzhou University of Science and Technology',
 'University of San Francisco, Quito',
 'University of South-Eastern Norway',
 'Westlake University',
 'Wuhan Textile University']

In [169]:
student_grader.check("q22", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q22...
Great job! You passed all test cases for this question.


True

#### Project Question 23

Between the years *2023* and *2024*, which institution had the **third largest** absolute change in its `World Rank`?

Your output **must** be a **string** representing the name of the institution with the **third greatest absolute difference** between its `World Rank` in 2023 and 2024. You **should** use the DataFrame `institutions_df` to answer this question. Feel free to add a new column to the DataFrame, so long as you do not modify the existing data.

Points possible: 4.0

In [172]:
abs_change = (institutions_df["2023 ranking"] - institutions_df["2024 ranking"]).abs()
third_most_change_inst = institutions_df.loc[abs_change.sort_values(ascending=False).index[2], "Institution"]
third_most_change_inst


'University of Franche-Comté'

In [173]:
student_grader.check("q23", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q23...
Great job! You passed all test cases for this question.


True

#### Project Question 24

For all the three years, find the **number** of institutions that **improved** their `World Rank` between **each year** by **at least** 5 ranks.

Your output **must** be an **integer** representing the number of institutions whose `World Rank` **decreased** each year by **at least** 5 ranks. You **should** use the DataFrame `institutions_df` to answer this question.

Points possible: 4.0

In [180]:
cond1 = institutions_df["2024 ranking"] <= institutions_df["2023 ranking"] - 5
cond2 = institutions_df["2025 ranking"] <= institutions_df["2024 ranking"] - 5
five_improved = len(institutions_df[cond1 & cond2])
five_improved


438

In [181]:
student_grader.check("q24", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q24...
Great job! You passed all test cases for this question.


True

#### Project Question 25

In the `Year` *2023*, **list** the institutions which do **not** feature in the **top** *50* in the world based on `World Rank`, but have a `Employability Rank` **less than or equal** to *25*.

Your output **must** be a **list** of institutions. **Order** does **not** matter. You **should** use the `year_2023_ranking_df` DataFrame that you created in Project Question 21 to answer this question.

Points possible: 4.0

In [178]:
cond1 = year_2023_ranking_df["World Rank"] > 50
cond2 = year_2023_ranking_df["Employability Rank"] <= 25
only_top_employability = list(year_2023_ranking_df[cond1 & cond2]["Institution"])
only_top_employability


['Keio University',
 'INSEAD',
 'Institut national du service public',
 'China Europe International Business School',
 'HEC Paris',
 'École des ponts ParisTech',
 'International Institute for Management Development',
 'Indian Institute of Management Ahmedabad',
 'Stockholm School of Economics',
 'Hitotsubashi University',
 'Central Party School of the Communist Party of China',
 'Graduate School, GRINM']

In [179]:
student_grader.check("q25", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q25...
Great job! You passed all test cases for this question.


True

#### Project Question 26

**List** the universities which ranked in the **top** 50 of world rankings (`World Rank`) in the `Year` *2023* but **failed** to do so in the `Year` *2025*.

Your output **must** be a **list** of institutions. The **order** does **not** matter. You **must** use the `year_2023_ranking_df` and `year_2025_ranking_df` DataFrames that you created in Project Question 21 to answer this question.

**Hints:**
1. There could be institutions that are ranked in the **top** 50 in *2023* but do not feature in *2025* at all; you still want to include them in your list.
2. You can use `sort_values` and `.iloc` to identify the **top** 50 institutions.
3. Given two *sets* `A` and `B`, you can find the elements which are in `A` but not in `B` using `A - B`. For example,
   <div style="height:6px;"></div>
   
    ```python
    set_A = {10, 20, 30, 40, 50}
    set_B = {20, 40, 70}
    set_A - set_B == {10, 30, 50} # elements which are in set_A but not in set_B
    ```

Points possible: 4.0

In [184]:
set_2023 = set(year_2023_ranking_df[year_2023_ranking_df["World Rank"] <= 50]["Institution"])
set_2025 = set(year_2025_ranking_df[year_2025_ranking_df["World Rank"] <= 50]["Institution"])
top_50_only_2023 = list(set_2023 - set_2025)
top_50_only_2023


['University of Manchester', 'University of Edinburgh']

In [185]:
student_grader.check("q26", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q26...
Great job! You passed all test cases for this question.


True

#### Project Question 27

**List** the countries which have **at least** *5* and **at most** *10* (both bounds inclusive) institutions featuring in the **top** *100* of world rankings (`World Rank`) in the `Year` *2023*.

Your output **must** be a **list**.

**Hints:**

1. In a `DataFrame`, to find the **number** of times each unique value in a column repeats, you can use the `value_counts` method. For example,
    ``` python
    rankings["Country"].value_counts()
    ```
    <div style="height:6px;"></div>
    
    would output a pandas `Series` with the **indices** being the unique values of `Country` and the **values** being the **number** of times each country has featured in the `rankings` DataFrame. You can find the documentation [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.dataframe.value_counts.html) or by using the `help` function in a separate cell. You can adapt this code to find the number of institutions from each country that features in the `Year` *2023*.
    <div style="height:5px;"></div>
2. Just like with **DataFrames**, you can use Boolean indexing on **Series**. For example, try something like this in a separate cell below:

    ```python
    a = pd.Series([100, 200, 300])
    a[a > 100]
    ```
    <div style="height:12px;"></div>
3. You can extract the **indices** of a `Series` `s` with `s.index`.

Points possible: 4.0

In [190]:
subset_2023 = year_2023_ranking_df[year_2023_ranking_df["World Rank"] <= 100]
counts = subset_2023["Country"].value_counts()
almost_top_countries = list(counts[(counts >= 5) & (counts <= 10)].index)
almost_top_countries

['United Kingdom', 'China', 'Germany', 'France']

In [191]:
student_grader.check("q27", should_get_llm_feedback=False)

Make sure you saved the notebook before running this cell. Running check for q27...
Great job! You passed all test cases for this question.


True

### Submission

Make sure you have run all cells in your notebook in order before submitting on Gradescope. Your notebook should not contain any uncaught Exceptions, otherwise the Gradescope autograder will not give you any points.